# SpaCET deconvolution — PanopTILs and HEST

Runs **SpaCET** (an unsupervised, reference-based spatial deconvolution method)
on Path2Space expression to estimate cell-type proportions per spot. This is
the upstream R step that produces the SpaCET predictions used by the
Python notebooks `01` (PanopTILs) and `02` (HEST).

- **PanopTILs** — SpaCET on Path2Space-inferred expression of the 1,317 TCGA
  archival-slide ROIs.
- **HEST** — SpaCET on both Path2Space-inferred and measured Visium expression
  for the two annotated slides (TENX13, TENX39).

Inputs and outputs are stored as compact `.qs` files in `../data/spacet/`.
Inferred expression is in log10 space, so `10^x - 1` is applied before
deconvolution; measured counts are used as-is. Adapted from
`SpaCET_pipline_panoptil_slurm_R.R` and `SpaCET_pipline_slurm_R.r`.

## Setup

`SpaCET.deconvolution_new()` (in `../lib/new_SpaCET.R`) is a thin wrapper around
the SpaCET package that deconvolves a gene-by-spot matrix directly.

In [1]:
suppressMessages({
    library(Matrix)
    library(SpaCET)
    library(qs)
})
source("../lib/new_SpaCET.R")

SPACET <- "../data/spacet/"

# Deconvolve one gene-by-spot matrix: read input qs, run SpaCET, write output qs.
run_spacet <- function(in_qs, out_qs, undo_log) {
    expr <- qread(file.path(SPACET, in_qs))
    if (undo_log) expr@x <- 10^expr@x - 1          # inferred expression is log10(x+1)
    prop <- as.data.frame(SpaCET.deconvolution_new(expr, "BRCA", coreNo = 12))
    qsave(prop, file.path(SPACET, out_qs))
    cat(sprintf("%-38s %d cell types x %d spots\n", out_qs, nrow(prop), ncol(prop)))
    prop
}

## PanopTILs

SpaCET on the Path2Space-inferred expression of the PanopTILs ROIs.

In [2]:
panoptils <- run_spacet("panoptils_inferred_expression.qs",
                        "panoptils_spacet_proportions.qs", undo_log = TRUE)
panoptils[1:6, 1:4]

[1] "Stage 1. Infer malignant cell fraction."
[1] "Stage 1 - Step 1. Clustering."
[1] "Stage 1 - Step 2. Find tumor clusters."
[1] "                  > Use CNA signature: BRCA"
[1] "Stage 1 - Step 3. Infer malignant cells."
[1] "Stage 2. Hierarchically deconvolve non-malignant cell fraction."
[1] "Stage 2 - Level 1. Estimate the major lineage."
[1] "Stage 2 - Level 2. Estimate the sub lineage."
panoptils_spacet_proportions.qs        35 cell types x 1317 spots


,TCGA-AR-A0TU-DX1_left-89903_top-23884_bottom-25129_right-91148.png,TCGA-AR-A0TU-DX1_left-89643_top-23896_bottom-25141_right-90888.png,TCGA-AR-A0TU-DX1_left-88624_top-23113_bottom-24358_right-89869.png,TCGA-AR-A0TU-DX1_left-90153_top-24126_bottom-25371_right-91398.png
,<dbl>,<dbl>,<dbl>,<dbl>
Malignant,4.213625e-01,0.5433137090,0.434819176,5.609756e-01
CAF,2.549903e-01,0.1811030690,0.171122524,1.289369e-01
Endothelial,5.966323e-02,0.0399644403,0.059105098,4.796986e-02
Plasma,2.847809e-02,0.0282875797,0.046731989,3.506294e-02
B cell,1.885296e-02,0.0161499703,0.015841005,2.276059e-02
T CD4,3.891356e-06,0.0008101337,0.002821576,7.448845e-05


## HEST

SpaCET on both inferred and measured expression for TENX13 and TENX39.

In [3]:
jobs <- list(
    c("hest_TENX13_inferred_expression.qs", "hest_TENX13_spacet_inferred.qs", TRUE),
    c("hest_TENX39_inferred_expression.qs", "hest_TENX39_spacet_inferred.qs", TRUE),
    c("hest_TENX13_measured_expression.qs", "hest_TENX13_spacet_measured.qs", FALSE),
    c("hest_TENX39_measured_expression.qs", "hest_TENX39_spacet_measured.qs", FALSE)
)
for (j in jobs) run_spacet(j[1], j[2], as.logical(j[3]))

[1] "Stage 1. Infer malignant cell fraction."
[1] "Stage 1 - Step 1. Clustering."
[1] "Stage 1 - Step 2. Find tumor clusters."
[1] "                  > Use CNA signature: BRCA"
[1] "Stage 1 - Step 3. Infer malignant cells."
[1] "Stage 2. Hierarchically deconvolve non-malignant cell fraction."
[1] "Stage 2 - Level 1. Estimate the major lineage."
[1] "Stage 2 - Level 2. Estimate the sub lineage."
hest_TENX13_spacet_inferred.qs         35 cell types x 3784 spots
[1] "Stage 1. Infer malignant cell fraction."
[1] "Stage 1 - Step 1. Clustering."
[1] "Stage 1 - Step 2. Find tumor clusters."
[1] "                  > Use CNA signature: BRCA"
[1] "Stage 1 - Step 3. Infer malignant cells."
[1] "Stage 2. Hierarchically deconvolve non-malignant cell fraction."
[1] "Stage 2 - Level 1. Estimate the major lineage."
[1] "Stage 2 - Level 2. Estimate the sub lineage."
hest_TENX39_spacet_inferred.qs         35 cell types x 2454 spots
[1] "Stage 1. Infer malignant cell fraction."
[1] "Stage 1 - Step 1. Clu

## Outputs

Each `*_spacet_*.qs` is a 35-cell-type by spot proportion matrix. The Python
notebooks collapse the 35 fine types into cancer / lymphocyte / stromal classes
and score them against pathologist annotations.

In [4]:
spacet_outputs <- list.files(SPACET, pattern = "spacet.*\\.qs$|proportions\\.qs$")
for (f in spacet_outputs) {
    d <- qread(file.path(SPACET, f))
    cat(sprintf("%-38s %d x %d\n", f, nrow(d), ncol(d)))
}

hest_TENX13_spacet_inferred.qs         35 x 3784
hest_TENX13_spacet_measured.qs         35 x 3789
hest_TENX39_spacet_inferred.qs         35 x 2454
hest_TENX39_spacet_measured.qs         35 x 2518
panoptils_spacet_proportions.qs        35 x 1317
